In [ ]:
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
import seaborn as sns

from emu_renewal.inputs import OUTPUTS_PATH, get_country_mobility
from emu_renewal.outputs import load_targets, get_country_analyses

In [ ]:
def get_collated_likes(country, job_path):
    collated_likes = {}
    country_path = job_path / country
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    for a in analyses:
        idata = az.from_netcdf(country_path / a / "idata_filtered.nc")
        total_like = -idata["sample_stats"]["lp"].to_pandas().T
        all_likes = pd.concat([idata["log_likelihood"].to_dataframe().reset_index(), total_like.melt()["value"]], axis=1)
        all_likes = all_likes.rename(columns={"value": "total_ll"})
        collated_likes[a] = all_likes
    return collated_likes
    
def get_like_components_df(collated_likes):
    components = collated_likes["no_mob"].columns[2:]
    like_comps_dfs = pd.DataFrame(columns=pd.MultiIndex.from_product([components, collated_likes.keys()]))
    for a in collated_likes:
        for c in components:
            like_comps_dfs[c, a] = collated_likes[a][c]
    return like_comps_dfs

def get_param_dist_df(country, job_path, param_name):
    country_path = job_path / country
    out_df = pd.DataFrame()
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    for a in analyses:
        idata = az.from_netcdf(country_path / a / "idata_filtered.nc")
        out_df[a] = idata["posterior"][param_name].to_dataframe()
    return out_df

def plot_country_procs_by_analysis(procs, mobility):
    proc_fig, axes = plt.subplots(3, 2, figsize=[11, 14], sharey=True)
    flat_axes = axes.ravel()
    for a, analysis in enumerate(procs.keys()):
        ax = flat_axes[a]
        quantiles = procs[analysis].quantile([0.05, 0.5, 0.95], axis=1).T
        ax.plot(quantiles[0.5], color="k")
        ax.fill_between(procs[analysis].index, quantiles[0.05], quantiles[0.95], alpha=0.4)
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
        ax.set_title(analysis)
        ax.set_yscale("log")
    flat_axes[-1].plot(mobility)
    plt.setp(flat_axes[-1].xaxis.get_majorticklabels(), rotation=70)
    proc_fig.suptitle(country, fontsize=15)
    proc_fig.tight_layout()

def plot_country_like_components(like_comps_dfs, proc_disp_samples, shared_disp_samples):
    comp_fig, axes = plt.subplots(4, 2, figsize=[11, 14])
    flat_axes = axes.ravel()
    for t, targ in enumerate(set(like_comps_dfs.columns.get_level_values(0))):
        ax = sns.kdeplot(like_comps_dfs[targ], fill=True, ax=flat_axes[t])
        ax.set_title(targ)
        flat_axes[t].get_legend().remove()
    sns.kdeplot(shared_disp_samples, fill=True, ax=flat_axes[-2])
    flat_axes[-2].set_title("shared_dispersion")    
    sns.kdeplot(proc_disp_samples, fill=True, ax=flat_axes[-1])
    flat_axes[-1].set_title("proc_dispersion")
    comp_fig.suptitle(country, fontsize=15)
    comp_fig.tight_layout()

In [ ]:
job_path = OUTPUTS_PATH / "42780561"
countries = [p.parts[-1] for p in job_path.iterdir()]
procs = {}
collated_likes = {}
like_comps_dfs = {}
mobility = {}
proc_disp_samples = {}
fit_disp_samples = {}
spaghs = {}
targets = {}
for country in countries[:1]:

    country_path = job_path / country
    analyses = get_country_analyses(country_path)
    procs[country] = {a: pd.read_hdf(country_path / a / "spaghetti.h5")["process"] for a in analyses}
    
    collated_likes[country] = get_collated_likes(country, job_path)
    like_comps_dfs[country] = get_like_components_df(collated_likes[country])
    mobility[country] = get_country_mobility(country)
    proc_disp_samples[country] = get_param_dist_df(country, job_path, "dispersion_proc")
    fit_disp_samples[country] = get_param_dist_df(country, job_path, "shared_dispersion")

    country_path = job_path / country
    analyses = [a.parts[-1] for a in country_path.iterdir()]
    country_spaghs = [pd.read_hdf(country_path / a / "spaghetti.h5") for a in analyses]
    spaghs[country] = pd.concat(country_spaghs, keys=analyses, axis=1)
    targets[country] = load_targets(country_path / analyses[0])

In [ ]:
for country in countries[:1]:
    c_spaghs = spaghs[country]
    c_targs = targets[country]
    plot_country_procs_by_analysis(procs[country], mobility[country])
    plot_country_like_components(like_comps_dfs[country], proc_disp_samples[country], fit_disp_samples[country])